# 📗 Docker 기초: 남이 만든 이미지와 내 이미지를 컨테이너로 실행하기

<div class="day78-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">GCP VM:</strong> docker 명령은 VS Code Remote-SSH로 연 VM 창의 터미널에서 실행합니다. 부록 01(GCP VM 준비)을 먼저 마칩니다.<br><strong style="font-size:1em;color:#1f3a5f;">내 PC:</strong> 브라우저로 VM의 웹 페이지를 열고, 파일 복사와 git은 실습자료 폴더에서 연 터미널에서 실행합니다. 내 PC 터미널 명령은 macOS 터미널과 Windows PowerShell에서 똑같이 실행합니다. 이 교안에는 노트북 코드 셀이 없습니다.<br><strong style="font-size:1em;color:#1f3a5f;">GitHub:</strong> 5절에서 공개 저장소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>를 만들어 교안 01~03에서 이어 씁니다.</div>

#### **컨테이너(container)**

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">컴퓨터 한 대를 격리된 실행 환경 여러 개로 나눠 쓰는 기술을 <strong style="font-size:1.13em;color:#1f3a5f;">가상화(virtualization)</strong>라고 하고, 가상 머신과 컨테이너 두 방식이 있습니다. <strong style="font-size:1.13em;color:#1f3a5f;">컨테이너는 커널(하드웨어와 프로세스를 관리하는 운영체제의 핵심)을 컨테이너를 실행하는 컴퓨터와 함께 쓰고, 프로그램이 쓰는 파일, 프로세스, 네트워크만 따로 나눈 실행 환경</strong>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 컨테이너는 운영체제를 따로 부팅하지 않아 가볍고 거의 바로 뜹니다. 그래서 서빙 API를 서버에 배포하는 표준 방법으로 씁니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/01_vm_vs_container.png" width="1050" alt="왼쪽 가상 머신은 하드웨어 위의 하이퍼바이저가 VM마다 게스트 OS를 따로 부팅하고 그 위에 라이브러리와 앱을 올립니다. 오른쪽 컨테이너는 하드웨어 위의 호스트 OS(리눅스 커널)를 모든 컨테이너가 함께 쓰고, 컨테이너 엔진 위에 라이브러리와 앱만 올립니다.">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안은 먼저 Docker Hub에 올라온 남이 만든 이미지를 받아 GCP VM에서 실행하고, 내 PC 브라우저로 그 웹 서버에 접속합니다. 이어서 웹 페이지 한 장을 보내는 내 이미지를 Dockerfile로 만들어 실행합니다. 교안 01은 같은 방법으로 노무 상담 서빙 API의 이미지를 만듭니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/08_lesson00_goals.png" width="1050" alt="학습 목표 그림: 1 가상화와 이미지, 2 남이 만든 이미지를 Docker Hub에서 받아 실행, 3 Dockerfile로 내 이미지 만들기, 4 레이어 캐시(바뀐 단계부터 다시)">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day78-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span> (그림의 번호와 같습니다)

1. 가상 머신과 컨테이너를 비교하고, 이미지, 컨테이너, 레지스트리를 설명합니다.
2. Docker Hub의 이미지를 docker pull 로 받아 컨테이너로 실행하고, 내 PC 브라우저에서 접속합니다.
3. Dockerfile로 웹 페이지 이미지를 만들고(docker build) 컨테이너로 실행합니다(docker run).
4. 레이어 캐시가 다시 쓰이는 조건과, 자주 바뀌는 파일을 뒤에 복사하는 이유를 설명합니다.

</div>

## 1. 가상화와 컨테이너 이미지

### 1-1. 가상 머신과 컨테이너

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">하이퍼바이저(hypervisor):</strong> 하드웨어를 가상 머신 여러 대에 나눠 주는 프로그램입니다. 가상 머신마다 게스트 OS가 자기 커널을 따로 부팅합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">컨테이너 엔진:</strong> 컨테이너를 만들고 실행하는 프로그램입니다. Docker Engine이 여기에 해당합니다.</li>
</ul>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">비교</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">가상 머신</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">컨테이너</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">시작</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">운영체제를 부팅하므로 수십 초에서 수 분</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">프로세스를 띄우는 것이라 거의 바로</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">크기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">운영체제 전체가 들어가 수 GB</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">커널 없이 운영체제 기본 파일, 런타임, 라이브러리, 앱만 들어가 수십에서 수백 MB</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">격리</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">커널까지 나뉘어 더 강함</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">커널을 함께 써서 가상 머신보다 약함</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">운영체제</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">아무 운영체제나 실행</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">호스트와 같은 종류(리눅스 컨테이너는 리눅스 커널 위에서만)</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">이 단원은 둘을 함께 씁니다. 리눅스 컨테이너는 리눅스 커널이 있어야 뜨므로, GCP에서 리눅스 가상 머신(VM) 한 대를 빌려 그 안에서 Docker로 컨테이너를 띄웁니다.</p>
</div>

### 1-2. 컨테이너 이미지

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">컨테이너 이미지(container image)</strong>는 <strong style="font-size:1.13em;color:#1f3a5f;">프로그램을 실행하는 데 필요한 파일(운영체제의 기본 파일, 파이썬 같은 언어 런타임, 라이브러리, 코드)과 실행 명령을 한 묶음으로 만든 것</strong>입니다. 컨테이너는 이 이미지를 실행한 것이고, 이미지 하나로 컨테이너를 여러 개 띄울 수 있습니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/00_image_what_why.png" width="1050" alt="왼쪽은 운영체제 기본 파일, 파이썬 런타임, 라이브러리, 앱 코드, 실행 명령을 층으로 쌓은 한 묶음의 이미지. 오른쪽 위는 코드만 옮기면 내 PC에서는 실행되고 서버에서는 라이브러리가 없어 멈추는 모습, 오른쪽 아래는 이미지로 옮기면 내 PC, GCP VM, GitHub Actions 러너에서 모두 실행되는 모습">

<p style="margin:6px 0;">코드 파일만 다른 컴퓨터로 옮기면 그 컴퓨터에 설치된 파이썬과 라이브러리에 따라 실행되기도 하고 멈추기도 합니다. 내 PC에서 되던 API가 서버에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ModuleNotFoundError: No module named 'fastapi'</code>로 멈추는 식입니다. 이미지는 실행에 필요한 것을 통째로 옮기므로, Docker가 있는 리눅스 컴퓨터라면 어디서나 같은 파일과 같은 명령으로 실행됩니다.</p>
<p style="margin:6px 0;">예를 들어 교안 01에서 만들 서빙 API 이미지에는 다섯 가지가 들어갑니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">운영체제 기본 파일:</strong> 데비안 리눅스의 기본 명령과 파일</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">언어 런타임:</strong> 파이썬 3.12</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">라이브러리:</strong> FastAPI, uvicorn 같은 서버 라이브러리</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">코드:</strong> 서빙 API 코드</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 명령:</strong> 서버를 띄우는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uvicorn</code> 명령</li>
</ul>
</div>

### 1-3. Docker 용어

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/02_docker_terms.png" width="1050" alt="Docker Engine 안에서 Dockerfile을 docker build해 이름:태그가 my-web:0.1인 이미지를 만들고, docker run으로 같은 이미지에서 컨테이너 두 개를 띄웁니다. 이미지는 push로 레지스트리에 올리고, 다른 컴퓨터가 pull로 받습니다.">

<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Docker Engine:</strong> 이미지를 빌드하고 컨테이너를 실행하는 프로그램입니다. 부록 01(GCP VM 준비)에서 VM에 설치했습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Dockerfile:</strong> 이미지를 만드는 명령을 위에서 아래 순서로 적은 파일입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker build</code>가 이 파일을 읽어 이미지를 만듭니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">컨테이너:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run</code>으로 이미지를 실행한 것입니다. 안에서 바꾼 파일은 그 컨테이너에만 남고 이미지는 바뀌지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">레지스트리(registry):</strong> 이미지를 올리고(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker push</code>) 받는(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker pull</code>) 저장소입니다. 가장 큰 공개 레지스트리가 <a style="color:#245fa0;text-decoration:underline;" href="https://hub.docker.com">Docker Hub</a>이고, GitHub의 GHCR(GitHub Container Registry)도 많이 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">태그(tag):</strong> 이미지 이름 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">:</code> 다음에 붙이는 버전 이름입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python:3.12-slim</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python</code> 이미지의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">3.12-slim</code> 버전입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">slim</code>은 실행에 꼭 필요한 파일만 넣어 크기를 줄인 버전이라는 뜻입니다.</li>
</ul>
</div>

## 2. 남이 만든 이미지 실행하기

### 2-1. Docker Hub에서 이미지 찾기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><a style="color:#245fa0;text-decoration:underline;" href="https://hub.docker.com">Docker Hub</a>에는 파이썬, 데이터베이스, 웹 서버처럼 자주 쓰는 프로그램의 이미지가 올라와 있습니다. 이름 옆에 <strong style="font-size:1.13em;color:#1f3a5f;">Docker Official Image</strong> 표시가 붙은 이미지는 Docker가 관리하는 공식 이미지라 믿고 쓸 수 있습니다. 로그인하지 않아도 받을 수 있습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">Docker Hub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://hub.docker.com/_/python</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">위쪽 검색창에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python</code>을 넣고, <strong style="font-size:1.13em;color:#1f3a5f;">Docker Official Image</strong> 표시가 붙은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python</code>을 누릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Tags</strong> 탭에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">3.12-slim</code> 같은 태그 목록을 봅니다. 이 교안은 2-2에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python:3.12-slim</code>, 2-3에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nginx:stable</code>을 받습니다.</li>
</ol>
</div>

### 2-2. 파이썬 이미지로 실행 환경 확인

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VS Code Remote-SSH로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>에 접속한 VM 창(부록 01(GCP VM 준비) 6-4)에서 터미널을 엽니다. Docker가 준비됐는지 먼저 보고, VM에 설치된 파이썬과 파이썬 이미지 안의 파이썬을 비교합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# Docker 명령 도구(Client)와 Docker Engine(Server)의 버전을 확인합니다
docker version

# VM 에 설치된 파이썬 버전을 봅니다
python3 --version

# Docker Hub 에서 파이썬 3.12 이미지를 받고, 받은 이미지를 목록에서 찾습니다
docker pull python:3.12-slim
docker images python

# 그 이미지로 컨테이너를 띄워 안에 든 파이썬 버전을 보고, 끝나면 컨테이너를 지웁니다
docker run --rm python:3.12-slim python --version
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker version</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">docker 명령 도구(Client)와 Docker Engine(Server)의 버전을 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker pull 이름:태그</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">레지스트리(Docker Hub)에서 이미지를 받아 VM에 저장합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker images 이름</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">VM에 저장된 이미지 중 그 이름의 목록을 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run 이미지 명령</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이미지로 컨테이너를 만들어, 이미지에 적힌 실행 명령 대신 뒤에 적은 명령을 실행합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--rm</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">명령이 끝나면 컨테이너를 바로 지웁니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker version</code>에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Client</code>와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Server</code> 두 묶음:</strong> 명령 도구가 Docker Engine에 연결되었습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Server</code> 대신 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">permission denied</code>가 보이면 부록 01(GCP VM 준비) 8-3대로 VM에 다시 접속합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">python3 --version</code>이 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Python 3.13...</code>(또는 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">command not found</code>), 컨테이너의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">python --version</code>이 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Python 3.12...</code>:</strong> 같은 VM 위에서도 컨테이너는 VM의 파이썬이 아니라 이미지에 든 파이썬을 씁니다. 1-2에서 본 &quot;실행에 필요한 것을 한 묶음으로&quot;가 이것입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker images</code>에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">python:3.12-slim</code> 줄:</strong> 받은 이미지가 VM에 남아 있어, 다음 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run</code>은 다시 받지 않습니다. 3절의 내 이미지도 이 이미지 위에 만듭니다.</li>
</ul>
</div>

### 2-3. 웹 서버 이미지로 내 PC 브라우저에서 접속

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">웹 서버 프로그램 nginx의 공식 이미지를 받아 띄우고, 내 PC 브라우저로 접속합니다. 서버 설치 없이 명령 한 줄로 웹 서버가 뜹니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# Docker Hub 에서 nginx 이미지를 받습니다
docker pull nginx:stable

# 받은 이미지로 웹 서버 컨테이너를 백그라운드에서 띄웁니다(VM 의 8080 -> 컨테이너의 80)
docker run -d --name web -p 8080:80 nginx:stable

# 실행 중인 컨테이너 목록을 봅니다
docker ps
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker pull nginx:stable</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Docker Hub에서 nginx 이미지를 받습니다(2-2와 같은 명령).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run ... nginx:stable</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">받아 둔 이미지로 컨테이너를 만들어 실행합니다. 이미지가 VM에 없으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run</code>이 먼저 받기도 합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-d</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너를 백그라운드(detached)에서 실행하고 터미널을 바로 돌려줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--name web</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너 이름입니다. 뒤 명령(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">logs</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">rm</code>)에서 이 이름으로 가리킵니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-p 8080:80</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><strong style="font-size:1.13em;color:#1f3a5f;">VM 포트:컨테이너 포트</strong>입니다. VM의 8080으로 들어온 연결을 컨테이너의 80(nginx가 받는 포트)으로 보냅니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker ps</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">실행 중인 컨테이너 목록을 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">이제 내 PC 브라우저 주소창에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://&lt;VM 외부 IP&gt;:8080</code>을 넣어 엽니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;VM 외부 IP&gt;</code>는 GCP 콘솔의 VM 목록에 보이는 외부 IP(실습자료 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>)입니다. 부록 01(GCP VM 준비) 5절의 방화벽 규칙이 8080 포트를 열어 두었습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker ps</code>의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">STATUS</code>가 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Up ...</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">PORTS</code>에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">8080-&gt;80</code>:</strong> 컨테이너가 실행 중이고 포트가 연결되었습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">브라우저에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Welcome to nginx!</code>:</strong> 내 PC의 요청이 VM의 8080 포트를 지나 컨테이너 안의 nginx까지 갔다가 돌아왔습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">브라우저가 오래 기다리다 실패하면:</strong> VM 목록의 외부 IP가 맞는지, VM이 실행 중인지 확인합니다.</li>
</ul>
<p style="margin:6px 0;">브라우저 요청이 컨테이너 로그에 남았는지 보고, 3절에서 같은 8080 포트를 쓰므로 이 컨테이너를 지웁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# 컨테이너의 로그(nginx 가 남긴 접속 기록)를 봅니다
docker logs web

# 컨테이너를 멈추고 지웁니다
docker rm -f web
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker logs 이름</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">그 컨테이너가 화면에 출력한 내용(로그)을 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker rm -f 이름</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너를 지웁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-f</code>는 실행 중이어도 멈춘 뒤 지웁니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker logs</code>에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">&quot;GET / HTTP/1.1&quot; 200</code> 줄:</strong> 내 PC 브라우저의 요청 기록입니다. 컨테이너를 지워도 받은 이미지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nginx:stable</code>은 VM에 남습니다.</li>
</ul>
</div>

## 3. 내 이미지 만들기

### 3-1. 실습 파일 올리기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">웹 페이지 한 장(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">index.html</code>)을 보내는 이미지를 만듭니다. 실습 폴더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker_basics/</code>의 두 파일을 내 PC에서 VM으로 복사합니다. <strong style="font-size:1.13em;color:#1f3a5f;">실습자료 폴더</strong>에서 연 내 PC 터미널에서 실행합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
scp -r day78_모델배포_Docker_CICD/교안_00_Docker_기초/docker_basics serving-vm:~/
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 인자</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">뜻</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">scp 보낼곳 받을곳</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><strong style="font-size:1.13em;color:#1f3a5f;">scp(secure copy)</strong>: SSH 연결로 내 PC와 서버 사이에 파일을 복사합니다. 앞이 보낼 파일, 뒤가 받을 자리입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-r</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">폴더를 안의 파일까지 통째로 복사합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm:~/</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Host 이름:경로</code> 모양의 받을 자리입니다. 부록 01(GCP VM 준비) 6-3에서 SSH 설정에 적은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>의 홈 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/</code>)입니다.</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">복사가 끝나면 파일마다 줄 끝에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">100%</code>가 보입니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · docker_basics/index.html</strong></div>

```html
<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <title>내 첫 Docker 이미지</title>
</head>
<body>
  <h1>내 첫 Docker 이미지</h1>
  <p>이 페이지는 GCP VM 의 컨테이너 안에서 파이썬 웹 서버가 보내고 있습니다.</p>
</body>
</html>
```

### 3-2. Dockerfile 읽기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker build</code>는 Dockerfile을 위에서 아래 순서로 실행합니다. 한 줄이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">명령 인자</code> 모양입니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · docker_basics/Dockerfile</strong></div>

```dockerfile
# 웹 페이지 한 장을 보내는 이미지(Docker 기초 교안)

# (1) 베이스 이미지: 파이썬 3.12 slim 버전
FROM python:3.12-slim

# (2) 작업 폴더
WORKDIR /app

# (3) 웹 페이지 파일을 이미지 안으로 복사합니다
COPY index.html .

# (4) 컨테이너가 쓰는 포트 표시
EXPOSE 8080

# (5) 실행 명령: 파이썬 내장 웹 서버로 /app 의 파일을 8080 포트에서 보냅니다
CMD ["python", "-m", "http.server", "8080"]
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이 파일에서</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">FROM</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">시작할 이미지(베이스 이미지)를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름:태그</code>로 정합니다. 첫 명령입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2-2에서 받은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python:3.12-slim</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">WORKDIR</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">뒤 명령이 실행될 이미지 안의 작업 폴더를 정합니다. 없으면 만듭니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">빌드 컨텍스트(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker build</code>에 준 폴더)의 파일을 이미지 안으로 복사합니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">index.html</code>을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app</code>으로</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">EXPOSE</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너가 쓰는 포트를 적어 두는 표시입니다. 포트를 열지는 않습니다(여는 것은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run -p</code>).</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">8080</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CMD</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><strong style="font-size:1.13em;color:#1f3a5f;">컨테이너를 시작할 때</strong> 실행할 명령입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파이썬 내장 웹 서버(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python -m http.server 8080</code>)</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">RUN</code>:</strong> 이 파일에는 없지만 자주 쓰는 명령입니다. <strong style="font-size:1.13em;color:#1f3a5f;">빌드할 때</strong> 명령을 실행하고 결과를 이미지에 남깁니다(예: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RUN pip install ...</code>로 라이브러리 설치). 교안 01의 Dockerfile에서 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">태그를 적는 이유:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">FROM python</code>처럼 태그를 빼면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code> 태그를 받습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code>는 새 버전이 나오면 바뀌므로, 같은 Dockerfile로도 빌드한 시점에 따라 다른 이미지가 나올 수 있습니다.</li>
</ul>
</div>

### 3-3. 빌드와 실행

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM에서 실습 폴더로 이동해 이미지를 만들고, 그 이미지로 컨테이너를 띄웁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
cd ~/docker_basics
pwd   # 끝이 docker_basics이면 맞습니다

# 지금 폴더(.)를 빌드 컨텍스트로 Dockerfile 을 읽어 이미지 my-web:0.1 을 만들고, 목록에서 찾습니다
docker build -t my-web:0.1 .
docker images my-web

# 내 이미지로 컨테이너를 띄우고(VM 의 8080 -> 컨테이너의 8080), 2초 뒤 VM 안에서 요청해 봅니다
docker run -d --name my-web -p 8080:8080 my-web:0.1
sleep 2
curl localhost:8080
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd 폴더</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pwd</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">그 폴더로 이동하고, 지금 폴더의 경로를 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker build -t 이름:태그 .</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.</code>)의 Dockerfile을 읽어 이미지를 만들고 이름과 태그를 붙입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sleep 2</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버가 뜰 때까지 2초 기다립니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">curl 주소</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">그 주소로 HTTP 요청을 보내고 응답 본문을 출력합니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">내 PC 브라우저에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://&lt;VM 외부 IP&gt;:8080</code>을 다시 엽니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">빌드 화면의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">[2/3] WORKDIR /app</code> 같은 줄:</strong> 3단계 중 2번째 단계입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">FROM</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">WORKDIR</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY</code>가 빌드 단계 하나씩(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[1/3]</code>~<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[3/3]</code>)이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">EXPOSE</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CMD</code>는 단계 없이 설정만 남깁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">FROM</code>은 2-2에서 받아 둔 이미지를 써서 빨리 끝납니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker images</code>에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">my-web:0.1</code> 줄:</strong> 내가 만든 이미지가 VM에 저장되었습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">curl</code> 출력과 브라우저에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">내 첫 Docker 이미지</code>:</strong> 내가 만든 이미지의 컨테이너가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">index.html</code>을 보냈습니다. 2-3의 nginx 대신 이번에는 내 이미지가 8080에서 받습니다.</li>
</ul>
<p style="margin:6px 0;">확인이 끝나면 컨테이너를 지웁니다. 교안 01의 서빙 API도 VM의 8080 포트를 씁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# my-web 컨테이너를 멈추고 지웁니다
docker rm -f my-web
```

## 4. 레이어와 캐시

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">레이어(layer):</strong> 이미지를 이루는 파일 변경 묶음입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RUN</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY</code>처럼 파일을 바꾸는 명령이 레이어를 하나씩 만들고, 이미지는 베이스 이미지 위에 이 레이어를 차례로 쌓은 것입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">빌드 캐시(build cache):</strong> 다시 빌드할 때 <strong style="font-size:1.13em;color:#1f3a5f;">앞 단계가 그대로이고 명령(복사할 파일의 내용까지)도 같으면</strong> 전에 만든 레이어를 다시 씁니다. 빌드 화면에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CACHED</code>로 보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">규칙:</strong> 한 단계가 다시 실행되면 그 뒤의 단계는 모두 다시 실행됩니다.</li>
</ul>
<p style="margin:6px 0;">라이브러리를 설치하는 아래 Dockerfile로 보면, 앱 코드(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app.py</code>)만 고쳐 다시 빌드할 때 오래 걸리는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RUN pip install</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CACHED</code>로 넘어가고 마지막 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY app.py</code>만 다시 실행됩니다. 설치보다 앱 코드를 <strong style="font-size:1.13em;color:#1f3a5f;">뒤에</strong> 복사했기 때문입니다. 원칙은 <strong style="font-size:1.13em;color:#1f3a5f;">거의 바뀌지 않는 것을 위에, 자주 바뀌는 것을 아래에</strong> 두는 것이고, 교안 01의 서빙 API Dockerfile이 이 순서를 따릅니다.</p>
</div>

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
CMD ["python", "app.py"]
```

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/09_dockerfile_layers.png" width="1050" alt="같은 파일로 다시 빌드하면 WORKDIR부터 COPY app.py까지 모두 CACHED, app.py만 고치고 다시 빌드하면 WORKDIR, COPY requirements.txt, RUN pip install은 CACHED이고 COPY app.py만 다시 실행">

## 5. 다음 교안 준비

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01~03이 쓰는 배포 저장소와 GitHub 읽기 토큰을 준비합니다.</p>
</div>

### 5-1. 배포 저장소 serving-deploy 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01~03은 내 GitHub의 공개 저장소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code> 하나를 이어 씁니다. 일차 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">배포_저장소/serving-deploy/</code>가 그 템플릿이고, 폴더마다 어느 교안에서 쓰는지는 템플릿의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">README.md</code>에 있습니다. 공개 저장소라야 VM이 로그인 없이 받아 갑니다(키는 저장소에 넣지 않습니다).</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/new</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">오른쪽 위 <strong style="font-size:1.13em;color:#1f3a5f;">+</strong> 에서 <strong style="font-size:1.13em;color:#1f3a5f;">New repository</strong>를 누릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Repository name</strong>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>를 넣고 <strong style="font-size:1.13em;color:#1f3a5f;">Public</strong>을 고릅니다. README, .gitignore, license는 추가하지 않습니다(빈 저장소여야 첫 push가 거절되지 않습니다).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Create repository</strong>를 누릅니다.</li>
</ol>
</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">템플릿을 홈 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/serving-deploy</code>)로 복사해 내 저장소로 씁니다. 실습자료 폴더는 자료를 새로 받을 때 덮어쓰기 때문입니다. <strong style="font-size:1.13em;color:#1f3a5f;">저장소 수정, 커밋, push는 내 PC에서만 하고, VM은 받기(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">git clone</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">git pull</code>)만 합니다.</strong></p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">아래 블록의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">&lt;GitHub 아이디&gt;</code>를 내 GitHub 아이디로 바꾼 뒤 실행합니다.</strong> 바꾸지 않으면 터미널이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&gt;</code>를 기호로 읽어 오류가 납니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GitHub 아이디:</strong> 이메일 주소가 아니라 사용자 이름(username)입니다. 내 프로필 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/아이디</code>의 마지막 부분이고, GitHub 오른쪽 위 프로필 사진을 누르면 메뉴 맨 위에 보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예:</strong> 프로필 주소가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/octocat</code>이면 아이디는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">octocat</code>이고, 바꾼 줄은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git remote add origin https://github.com/octocat/serving-deploy.git</code>입니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
cp -r day78_모델배포_Docker_CICD/배포_저장소/serving-deploy ~/serving-deploy
pushd ~/serving-deploy
git init -b main
git add .
git commit -m "serving-deploy 템플릿 올리기"
git remote add origin https://github.com/<GitHub 아이디>/serving-deploy.git
git push -u origin main
popd
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cp -r 원본 대상</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">템플릿 폴더를 안의 파일까지 통째로 홈 폴더로 복사합니다(실습자료 안의 원본은 그대로 둡니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pushd 폴더</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 위치를 기억하고 그 폴더로 이동합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git init -b main</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 폴더를 git 저장소로 만들고 첫 브랜치 이름을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main</code>으로 정합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git add .</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 폴더의 모든 파일을 다음 커밋에 담습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git commit -m &quot;메시지&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">담은 파일을 메시지와 함께 기록(커밋) 하나로 남깁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git remote add origin 주소</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">GitHub 저장소 주소를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">origin</code>이라는 이름으로 등록합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git push -u origin main</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">커밋을 GitHub의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main</code> 브랜치로 올립니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-u</code>를 한 번 주면 다음부터 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git push</code>만 써도 같은 곳으로 올립니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">popd</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pushd</code>가 기억한 위치(실습자료 폴더)로 돌아옵니다.</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">GitHub 저장소 화면을 새로 고쳐 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy</code> 같은 폴더가 보이면 올라간 것입니다.</li>
<li style="margin:3px 0;">이 저장소의 파일을 고칠 때는 VS Code에서 <strong style="font-size:1.13em;color:#1f3a5f;">File &gt; New Window</strong>로 새 창을 열고 <strong style="font-size:1.13em;color:#1f3a5f;">File &gt; Open Folder</strong>로 홈 폴더(macOS <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/Users/사용자이름</code>, Windows <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">C:\Users\사용자이름</code>)의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>를 엽니다.</li>
</ul>
</div>

### 5-2. .env에 GitHub 아이디와 읽기 토큰 적기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">필요한 이유:</strong> 교안 01, 03과 과제의 노트북이 GitHub 아이디로 내 저장소를 찾아 Actions 실행 결과를 읽는데, 토큰 없이 읽으면 교실 인터넷 주소 하나에 시간당 60번 제한이 걸려 금방 막히기 때문입니다(읽기 전용 토큰은 학생마다 시간당 5,000번).</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/settings/personal-access-tokens/new</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">오른쪽 위 프로필 사진 &gt; <strong style="font-size:1.13em;color:#1f3a5f;">Settings</strong> &gt; 왼쪽 맨 아래 <strong style="font-size:1.13em;color:#1f3a5f;">Developer settings</strong> &gt; <strong style="font-size:1.13em;color:#1f3a5f;">Personal access tokens</strong> &gt; <strong style="font-size:1.13em;color:#1f3a5f;">Fine-grained tokens</strong> &gt; <strong style="font-size:1.13em;color:#1f3a5f;">Generate new token</strong>을 누릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Token name</strong>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day78-read</code>를 넣고, <strong style="font-size:1.13em;color:#1f3a5f;">Expiration</strong>은 30일로 둡니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Repository access</strong>에서 <strong style="font-size:1.13em;color:#1f3a5f;">Public repositories</strong>를 고릅니다(공개 저장소를 읽기만 하는 토큰이라 권한을 더 고르지 않습니다).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Generate token</strong>을 누르고 나온 토큰(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">github_pat_</code>로 시작)을 복사합니다. 이 화면을 닫으면 다시 볼 수 없습니다.</li>
</ol>
</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 PC의 VS Code에서 실습자료 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 열고 맨 아래에 두 줄을 더해 저장합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;GitHub 아이디&gt;</code>는 5-1에서 쓴 사용자 이름(이메일 아님), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;토큰&gt;</code>은 복사한 토큰으로 바꿉니다. 아이디가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">octocat</code>이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_OWNER=octocat</code>, 토큰 줄은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_READ_TOKEN=github_pat_</code>로 시작하는 한 줄입니다. 토큰은 노트북에 출력하지 않고 git에도 올리지 않습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · 실습자료/.env 에 더할 줄</strong></div>

```text
GITHUB_OWNER=<GitHub 아이디>
GITHUB_READ_TOKEN=<토큰>
```

### 5-3. VM에 저장소 받기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VS Code Remote-SSH로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>에 접속한 창에서 터미널을 열고 내 저장소를 받습니다. 공개 저장소라 로그인 없이 받습니다. <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">&lt;GitHub 아이디&gt;</code>를 5-1과 같이 바꾼 뒤 실행합니다.</strong></p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# 홈 폴더에 내 저장소를 받습니다(<GitHub 아이디> 를 먼저 바꿉니다)
cd ~
git clone https://github.com/<GitHub 아이디>/serving-deploy.git
cd ~/serving-deploy
pwd   # 끝이 serving-deploy이면 맞습니다
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git clone 주소</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">GitHub 저장소를 지금 폴더 아래에 통째로 받습니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">pwd</code> 출력의 끝이 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>:</strong> 저장소를 받아 그 폴더로 들어왔습니다. 교안 01은 이 폴더에서 시작합니다. VM은 교안 01에서 계속 쓰므로 켜 둡니다.</li>
</ul>
</div>